# Data Cleaning and Preprocessing
**AI101 Mini-Project, Group 13: Hit Song Popularity Prediction**
Author: Hemanshu Samadhan Borse (2026ESB1477, GitHub: xlent-h)

This notebook loads the Spotify Tracks dataset, inspects it, cleans it, and saves
`data/cleaned_dataset.csv` for the EDA and model notebooks.

It answers four questions:
1. How many rows and columns are there before cleaning?
2. Which columns have missing values?
3. How many records are duplicated?
4. What did cleaning change?

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 30)

# Works from the repo root or from the notebooks/ folder (relative paths only)
ROOT = Path("..") if Path("../data").exists() else Path(".")
DATA_PATH = ROOT / "data" / "dataset.csv"
CLEAN_PATH = ROOT / "data" / "cleaned_dataset.csv"
OUT_DIR = ROOT / "outputs"
OUT_DIR.mkdir(exist_ok=True)

## 1. Load the dataset and check its size

In [2]:
df_raw = pd.read_csv(DATA_PATH)
rows_before, cols_before = df_raw.shape
print(f"Rows before cleaning   : {rows_before:,}")
print(f"Columns before cleaning: {cols_before}")
df_raw.head()

Rows before cleaning   : 114,000
Columns before cleaning: 22


,Unnamed: 0.1,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [3]:
print(df_raw.dtypes)

Unnamed: 0.1          int64
Unnamed: 0            int64
track_id             object
artists              object
album_name           object
track_name           object
popularity            int64
duration_ms           int64
explicit               bool
danceability        float64
energy              float64
key                   int64
loudness            float64
mode                  int64
speechiness         float64
acousticness        float64
instrumentalness    float64
liveness            float64
valence             float64
tempo               float64
time_signature        int64
track_genre          object
dtype: object


## 2. Missing values

In [4]:
missing = df_raw.isna().sum()
missing = missing[missing > 0]
print("Columns with missing values:")
print(missing if len(missing) else "None")

Columns with missing values:
artists       1
album_name    1
track_name    1
dtype: int64


## 3. Duplicates
A `track_id` can repeat because the same song may be listed under several genres.
That matters later: if the same song lands in both the training and the test set,
the test score is too optimistic. The model notebook should split by `track_id`.

In [5]:
unnamed_cols = [c for c in df_raw.columns if c.startswith("Unnamed")]
base = df_raw.drop(columns=unnamed_cols)

full_dups = int(base.duplicated().sum())
repeated_id_rows = int(base.duplicated("track_id").sum())
same_id_same_genre = int(base.duplicated(["track_id", "track_genre"]).sum())

repeat = base[base.duplicated("track_id", keep=False)]
ids_multi_genre = int(repeat.groupby("track_id")["track_genre"].nunique().gt(1).sum())
ids_pop_differs = int(repeat.groupby("track_id")["popularity"].nunique().gt(1).sum())

print(f"Unnamed index columns            : {unnamed_cols}")
print(f"Number of genres                 : {base['track_genre'].nunique()}")
print(f"Fully identical rows             : {full_dups:,}")
print(f"Rows repeating an earlier track_id: {repeated_id_rows:,}")
print(f"Rows repeating track_id + genre  : {same_id_same_genre:,}")
print(f"track_ids listed in >1 genre     : {ids_multi_genre:,}")
print(f"track_ids with differing popularity: {ids_pop_differs:,}")

Unnamed index columns            : ['Unnamed: 0.1', 'Unnamed: 0']
Number of genres                 : 114
Fully identical rows             : 450
Rows repeating an earlier track_id: 24,259
Rows repeating track_id + genre  : 450
track_ids listed in >1 genre     : 16,299
track_ids with differing popularity: 720


## 4. Invalid records
A record is treated as invalid if:
- popularity is outside 0 to 100,
- danceability, energy, speechiness, acousticness, instrumentalness, liveness or valence is outside 0 to 1,
- duration is zero or negative, or tempo is zero or negative (a tempo of 0 means no beat was detected).

In [6]:
UNIT_RANGE = [c for c in ["danceability", "energy", "speechiness", "acousticness",
                          "instrumentalness", "liveness", "valence"] if c in base.columns]

def invalid_mask(d):
    bad = ~d["popularity"].between(0, 100)
    for c in UNIT_RANGE:
        bad |= ~d[c].between(0, 1)
    bad |= d["duration_ms"] <= 0
    bad |= d["tempo"] <= 0
    return bad

n_invalid = int(invalid_mask(base.dropna()).sum())
print(f"Invalid records (excluding rows with missing values): {n_invalid:,}")
print(f"Tracks with popularity = 0 (kept, but worth noting): {int((base['popularity'] == 0).sum()):,}")

Invalid records (excluding rows with missing values): 157
Tracks with popularity = 0 (kept, but worth noting): 16,020


## 5. Cleaning

In [7]:
df = df_raw.copy()
log = [("Original dataset", len(df), df.shape[1])]

# a) drop the unnamed index column(s)
df = df.drop(columns=unnamed_cols)
log.append(("Drop unnamed index column(s)", len(df), df.shape[1]))

# b) drop rows with missing values
df = df.dropna()
log.append(("Drop rows with missing values", len(df), df.shape[1]))

# c) drop fully identical rows
df["track_genre"] = df["track_genre"].str.strip()
df = df.drop_duplicates()
log.append(("Drop fully identical rows", len(df), df.shape[1]))

# d) drop repeats of the same track in the same genre (keep the first)
df = df.drop_duplicates(subset=["track_id", "track_genre"], keep="first")
log.append(("Drop repeated track_id within the same genre", len(df), df.shape[1]))

# e) drop invalid records
df = df[~invalid_mask(df)]
log.append(("Drop invalid records", len(df), df.shape[1]))

# f) make 'explicit' numeric (0/1) so the model can use it
if df["explicit"].dtype == bool:
    df["explicit"] = df["explicit"].astype(int)

df = df.reset_index(drop=True)

summary = pd.DataFrame(log, columns=["step", "rows_after", "columns_after"])
summary["rows_removed"] = (-summary["rows_after"].diff()).fillna(0).astype(int)
summary

,step,rows_after,columns_after,rows_removed
0,Original dataset,114000,22,0
1,Drop unnamed index column(s),114000,20,0
2,Drop rows with missing values,113999,20,1
3,Drop fully identical rows,113549,20,450
4,Drop repeated track_id within the same genre,113549,20,0
5,Drop invalid records,113392,20,157


## 6. Check the result and save

In [8]:
rows_after, cols_after = df.shape
print(f"Rows    : {rows_before:,} -> {rows_after:,}  ({rows_before - rows_after:,} removed)")
print(f"Columns : {cols_before} -> {cols_after}")
print(f"Missing values left   : {int(df.isna().sum().sum())}")
print(f"Duplicate rows left   : {int(df.duplicated().sum())}")
print(f"Genres                : {df['track_genre'].nunique()}")
print(f"Unique track_ids      : {df['track_id'].nunique():,}")
print(f"track_ids in >1 genre : {int(df.groupby('track_id')['track_genre'].nunique().gt(1).sum()):,}")
print()
print(df["popularity"].describe().round(2))

Rows    : 114,000 -> 113,392  (608 removed)
Columns : 22 -> 20
Missing values left   : 0
Duplicate rows left   : 0
Genres                : 114
Unique track_ids      : 89,583
track_ids in >1 genre : 16,299

count    113392.00
mean         33.32
std          22.30
min           0.00
25%          17.00
50%          35.00
75%          50.00
max         100.00
Name: popularity, dtype: float64


In [9]:
df.to_csv(CLEAN_PATH, index=False)
summary.to_csv(OUT_DIR / "cleaning_summary.csv", index=False)
print(f"Saved {CLEAN_PATH} and {OUT_DIR / 'cleaning_summary.csv'}")

Saved data/cleaned_dataset.csv and outputs/cleaning_summary.csv


## Notes for the team
- Use `data/cleaned_dataset.csv` in the EDA, model and evaluation notebooks.
- `track_id` is kept on purpose. When splitting into train and test, group by `track_id`
  (for example with `GroupShuffleSplit`) so the same song is never in both sets.
- `track_genre` is still text. The model notebook must convert it to numbers (one-hot encoding).